<a href="https://colab.research.google.com/github/expaetra/CM3070_final_project/blob/master/12_weighted_titles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!git clone https://github.com/expaetra/CM3070_final_project.git
%cd CM3070_final_project

Cloning into 'CM3070_final_project'...
remote: Enumerating objects: 238, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 238 (delta 66), reused 47 (delta 17), pack-reused 129 (from 1)
Receiving objects: 100% (238/238), 71.10 MiB | 18.07 MiB/s, done.
Resolving deltas: 100% (104/104), done.
Updating files: 100% (63/63), done.
/content/CM3070_final_project


## Weighted titles

Since an improvement in perfromance was observed after addind titles to the dataset, this notebook attempts to assess whetherthe titles carry storng enough signal so that weighing them furher improves it.

In [22]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords

In [23]:
# load dataset

df = pd.read_csv("/content/arxiv_abstracts_and_titles.csv")
print(df.shape)
df.head()

(79141, 7)


,category,field,discipline,title,abstract,abstract_length,title_length
0,cs.LG,Machine Learning,Artificial Intelligence,how do lexical semantics affect translation an...,neural machine translation (nmt) systems aim t...,177,9
1,cs.LG,Machine Learning,Artificial Intelligence,barack partially supervised group robustness w...,while neural networks have shown remarkable su...,203,7
2,cs.LG,Machine Learning,Artificial Intelligence,a deep learning approach to integrate humanlev...,"in recent times, a large number of people have...",187,11
3,cs.LG,Machine Learning,Artificial Intelligence,croesus multistage processing and transactions...,emerging edge applications require both a fast...,194,10
4,cs.LG,Machine Learning,Artificial Intelligence,representation topology divergence a method fo...,comparison of data representations is a comple...,133,10


In [24]:
# apply cap

cap = 2500

df_capped = (
    df.groupby('discipline', group_keys=False)[df.columns]
      .apply(lambda x: x.sample(n=min(len(x), cap), random_state=42))
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)


In [25]:
# apply cap

cap = 2500

# combine title + abstract

df_capped["text"] = df_capped["title"] + " " +df_capped["abstract"]
df["text"] = df["title"] + " " + df["abstract"]

# discipline
X_text_disc = df_capped["text"].astype(str)
y_disc = df_capped["discipline"]

# field
X_text_field = df["text"].astype(str)
y_field = df["field"]

In [26]:
# stopwords

nltk.download("stopwords")

custom_stops = {
    'based', 'paper', 'show', 'results', 'problem', 'using', 'approach',
    'proposed', 'method', 'methods', 'propose', 'present', 'work',
    'used', 'use', 'two', 'one', 'new', 'also', 'shows', 'however',
    'provide', 'study', 'task', 'tasks', 'different', 'high', 'given'
}
stop_words = set(stopwords.words('english')).union(custom_stops)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
# prepare data

# discipline
y_disc = df_capped["discipline"]

# field
y_field = df["field"]

# title weights
title_weights =[1, 2, 3, 4, 5]

In [28]:
def evaluate_result(y_test, y_pred, task, model_name, representation):

    accuracy = accuracy_score(y_test, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='macro', zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='weighted',zero_division=0
    )

    return {
        "Task": task,
        "Model": model_name,
        "Representation": representation,
        "Accuracy": round(accuracy, 3),
        "Macro Precision": round(macro_p,3),
        "Macro Recall": round(macro_r, 3),
        "Macro F1": round(macro_f1, 3),
        "Weighted Precision": round(weighted_p, 3),
        "Weighted Recall": round(weighted_r, 3),
        "Weighted F1": round(weighted_f1, 3)
    }

In [29]:
results = []

for title_weight in title_weights:

    # combine title + abstract
    df_capped["text"] = (df_capped["title"] + " ") * title_weight + df_capped["abstract"]
    df["text"] = (df["title"] + " ") * title_weight + df["abstract"]

    # disciplne
    X_text_disc = df_capped["text"].astype(str)
    # field
    X_text_field = df["text"].astype(str)

    # train/test split:

    # discipline
    X_train_text_disc,X_test_text_disc, y_disc_train, y_disc_test = train_test_split(
        X_text_disc,
        y_disc,
        test_size=0.2,
        random_state=42,
        stratify=y_disc
    )

    # field
    X_train_text_field, X_test_text_field, y_field_train, y_field_test = train_test_split(
        X_text_field,
        y_field,
        test_size= 0.2,
        random_state=42,
        stratify=y_field
    )

    # TF-IDF
    vectorizer_disc = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    max_features=15000
    )

    vectorizer_field = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    max_features=15000
    )

    # fit + transform
    X_train_disc = vectorizer_disc.fit_transform(X_train_text_disc)
    X_test_disc = vectorizer_disc.transform(X_test_text_disc)

    X_train_field = vectorizer_field.fit_transform(X_train_text_field)
    X_test_field =vectorizer_field.transform(X_test_text_field)

    # train models
    disc_model = LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight='balanced',
    solver='liblinear'
    )
    disc_model.fit(X_train_disc, y_disc_train)

    field_model = LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight='balanced',
    solver='liblinear'
    )
    field_model.fit(X_train_field, y_field_train)

    # predictions
    y_disc_pred = disc_model.predict(X_test_disc)
    y_field_pred = field_model.predict(X_test_field)

    results.append(
        evaluate_result(
            y_disc_test,
            y_disc_pred,
            task="Discipline",
            model_name="Logistic Regression",
            representation=f"TF-IDF word unigrams (title {title_weight}x + abstract, capped discipline)"
        )
    )
    results.append(
        evaluate_result(
            y_field_test,
            y_field_pred,
            task="Field",
            model_name="Logistic Regression",
            representation=f"TF-IDF word unigrams (title {title_weight}x + abstract, uncapped field)"
        )
    )

In [30]:
results_df = pd.DataFrame(results)
results_df

,Task,Model,Representation,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0,Discipline,Logistic Regression,"TF-IDF word unigrams (title 1x + abstract, cap...",0.634,0.630,0.632,0.629,0.631,0.634,0.630
1,Field,Logistic Regression,"TF-IDF word unigrams (title 1x + abstract, unc...",0.549,0.538,0.549,0.541,0.537,0.549,0.541
2,Discipline,Logistic Regression,"TF-IDF word unigrams (title 2x + abstract, cap...",0.633,0.631,0.632,0.629,0.631,0.633,0.630
3,Field,Logistic Regression,"TF-IDF word unigrams (title 2x + abstract, unc...",0.551,0.539,0.551,0.543,0.539,0.551,0.543
4,Discipline,Logistic Regression,"TF-IDF word unigrams (title 3x + abstract, cap...",0.634,0.631,0.633,0.630,0.631,0.634,0.631
5,Field,Logistic Regression,"TF-IDF word unigrams (title 3x + abstract, unc...",0.548,0.537,0.549,0.541,0.537,0.548,0.540
6,Discipline,Logistic Regression,"TF-IDF word unigrams (title 4x + abstract, cap...",0.631,0.628,0.630,0.627,0.628,0.631,0.628
7,Field,Logistic Regression,"TF-IDF word unigrams (title 4x + abstract, unc...",0.546,0.535,0.546,0.538,0.535,0.546,0.538
8,Discipline,Logistic Regression,"TF-IDF word unigrams (title 5x + abstract, cap...",0.626,0.624,0.625,0.622,0.624,0.626,0.623
9,Field,Logistic Regression,"TF-IDF word unigrams (title 5x + abstract, unc...",0.543,0.532,0.543,0.535,0.531,0.543,0.535


In [31]:
results_df.sort_values(["Task", "Macro F1"], ascending=[True, False])

,Task,Model,Representation,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
4,Discipline,Logistic Regression,"TF-IDF word unigrams (title 3x + abstract, cap...",0.634,0.631,0.633,0.630,0.631,0.634,0.631
0,Discipline,Logistic Regression,"TF-IDF word unigrams (title 1x + abstract, cap...",0.634,0.630,0.632,0.629,0.631,0.634,0.630
2,Discipline,Logistic Regression,"TF-IDF word unigrams (title 2x + abstract, cap...",0.633,0.631,0.632,0.629,0.631,0.633,0.630
6,Discipline,Logistic Regression,"TF-IDF word unigrams (title 4x + abstract, cap...",0.631,0.628,0.630,0.627,0.628,0.631,0.628
8,Discipline,Logistic Regression,"TF-IDF word unigrams (title 5x + abstract, cap...",0.626,0.624,0.625,0.622,0.624,0.626,0.623
3,Field,Logistic Regression,"TF-IDF word unigrams (title 2x + abstract, unc...",0.551,0.539,0.551,0.543,0.539,0.551,0.543
1,Field,Logistic Regression,"TF-IDF word unigrams (title 1x + abstract, unc...",0.549,0.538,0.549,0.541,0.537,0.549,0.541
5,Field,Logistic Regression,"TF-IDF word unigrams (title 3x + abstract, unc...",0.548,0.537,0.549,0.541,0.537,0.548,0.540
7,Field,Logistic Regression,"TF-IDF word unigrams (title 4x + abstract, unc...",0.546,0.535,0.546,0.538,0.535,0.546,0.538
9,Field,Logistic Regression,"TF-IDF word unigrams (title 5x + abstract, unc...",0.543,0.532,0.543,0.535,0.531,0.543,0.535


In [32]:
# save

os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)
df_capped.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/weighted_titles.csv",
    index=False
)

results_df.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/weighted_titles_results.csv",
    index=False
)

print("Saved")

Saved
